# Generate Out-of-Ordinary 311 Complaint Rankings

This notebook ranks complaint categories that are unusually high relative to the fitted reported-pressure model.

Outputs:
- top anomalous complaints overall (latest month)
- top anomalous complaints per neighborhood (latest month)
- top anomalous complaints per neighborhood (recent window)


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    sys.path.insert(0, str(cwd.parent))
else:
    sys.path.insert(0, str(cwd))

In [ ]:
import numpy as np
import pandas as pd
import arviz as az

from helpers import prep_the_data, load_idata

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

In [ ]:
IDATA_PATH = "../data/processed/models/model_urban_pressure_idata.nc"
DATA_PATH = "../data/processed/features/complaints_311_nta_category.parquet"
OUTPUT_DIR = Path("../data/processed/rankings")

TOP_N_PER_NTA = 5
RECENT_MONTHS = 3
MIN_EXPECTED_FOR_RATIO = 1.0

In [ ]:
idata = load_idata(IDATA_PATH)
df_raw = pd.read_parquet(DATA_PATH)
df = prep_the_data(df_raw)
df["complaint_date"] = pd.to_datetime(df["complaint_date"], errors="coerce")

if "level_2" not in df.columns and "category" in df.columns:
    df["level_2"] = df["category"]
if "category" not in df.columns and "level_2" in df.columns:
    df["category"] = df["level_2"]

print("idata posterior dims:", idata.posterior.dims)
print("raw rows:", len(df))

## Build Validation Grid Aligned to Model Obs

In [ ]:
required_coords = ["nta", "month", "category"]
for c in required_coords:
    if c not in idata.posterior.coords:
        raise KeyError(f"Missing required coordinate '{c}' in idata.posterior")

nta_vals = pd.Index(idata.posterior.coords["nta"].values, name="nta_id")
month_vals = pd.to_datetime(pd.Index(idata.posterior.coords["month"].values, name="month_start"))
cat_vals = pd.Index(idata.posterior.coords["category"].values, name="level_2")

obs_monthly = (
    df.assign(month_start=lambda d: d["complaint_date"].dt.to_period("M").dt.start_time)
    .groupby(["nta_id", "month_start", "level_2"], observed=True)["complaint_count"]
    .sum()
    .reset_index(name="y_obs")
)

model_grid = pd.MultiIndex.from_product(
    [nta_vals, month_vals, cat_vals],
    names=["nta_id", "month_start", "level_2"],
).to_frame(index=False)

model_df = model_grid.merge(obs_monthly, on=["nta_id", "month_start", "level_2"], how="left")
model_df["y_obs"] = model_df["y_obs"].fillna(0.0)

meta_cols = [c for c in ["nta_id", "nta_name", "borough", "level_1", "level_2"] if c in df.columns]
meta = df[meta_cols].drop_duplicates() if meta_cols else None
if meta is not None and len(meta_cols) > 1:
    merge_keys = [k for k in ["nta_id", "level_2"] if k in meta.columns]
    if merge_keys:
        model_df = model_df.merge(meta, on=merge_keys, how="left")

print("aligned model_df rows:", len(model_df))
print("posterior obs:", int(idata.posterior.sizes.get("obs", -1)))
display(model_df.head())

## Extract Posterior Mean/Uncertainty for Mu

In [ ]:
def posterior_obs_samples(idata, var_name):
    arr = idata.posterior[var_name]
    stacked = arr.stack(sample=("chain", "draw"))
    if "obs" in stacked.dims:
        return stacked.transpose("obs", "sample").values
    obs_dims = [d for d in stacked.dims if d != "sample"]
    if len(obs_dims) != 1:
        raise ValueError(f"Cannot infer obs dimension for {var_name}: {stacked.dims}")
    return stacked.transpose(obs_dims[0], "sample").values

if "mu" not in idata.posterior.data_vars:
    raise KeyError("Expected posterior variable 'mu' not found")

mu_samples = posterior_obs_samples(idata, "mu")
mu_mean = mu_samples.mean(axis=1)
mu_sd = mu_samples.std(axis=1)
mu_p10 = np.percentile(mu_samples, 10, axis=1)
mu_p90 = np.percentile(mu_samples, 90, axis=1)

if len(model_df) != len(mu_mean):
    raise ValueError(f"Alignment error: model_df rows={len(model_df)}, posterior obs={len(mu_mean)}")

model_df["mu_mean"] = mu_mean
model_df["mu_sd"] = mu_sd
model_df["mu_p10"] = mu_p10
model_df["mu_p90"] = mu_p90

## Compute Out-of-Ordinary Scores

In [ ]:
eps = 1e-9
model_df["residual"] = model_df["y_obs"] - model_df["mu_mean"]
model_df["abs_residual"] = model_df["residual"].abs()
model_df["pressure_ratio"] = model_df["y_obs"] / np.maximum(model_df["mu_mean"], MIN_EXPECTED_FOR_RATIO)
model_df["log_pressure_ratio"] = np.log1p(model_df["y_obs"]) - np.log1p(model_df["mu_mean"])
model_df["z_score"] = model_df["residual"] / np.maximum(model_df["mu_sd"], eps)
model_df["is_above_p90"] = model_df["y_obs"] > model_df["mu_p90"]

display(
    model_df[["nta_id", "month_start", "level_2", "y_obs", "mu_mean", "residual", "pressure_ratio", "z_score"]]
    .head(10)
)

## Ranking Tables

In [ ]:
latest_month = model_df["month_start"].max()
latest_df = model_df[model_df["month_start"] == latest_month].copy()

# Overall anomalies for latest month.
overall_latest = (
    latest_df.sort_values(["z_score", "residual", "pressure_ratio"], ascending=False)
    .reset_index(drop=True)
)

# Top-N anomalies per neighborhood for latest month.
per_nta_latest = (
    latest_df.sort_values(["nta_id", "z_score", "residual"], ascending=[True, False, False])
    .groupby("nta_id", observed=True)
    .head(TOP_N_PER_NTA)
    .reset_index(drop=True)
)
per_nta_latest["rank_within_nta"] = per_nta_latest.groupby("nta_id", observed=True).cumcount() + 1

# Recent window summary for stability.
recent_start = latest_month - pd.DateOffset(months=RECENT_MONTHS - 1)
recent_df = model_df[model_df["month_start"] >= recent_start].copy()
recent_summary = (
    recent_df.groupby(["nta_id", "level_2"], observed=True)
    .agg(
        months=("month_start", "nunique"),
        observed_total=("y_obs", "sum"),
        expected_total=("mu_mean", "sum"),
        residual_total=("residual", "sum"),
        mean_z=("z_score", "mean"),
        max_z=("z_score", "max"),
        pct_above_p90=("is_above_p90", "mean"),
    )
    .reset_index()
)
recent_summary["pressure_ratio_total"] = recent_summary["observed_total"] / np.maximum(recent_summary["expected_total"], MIN_EXPECTED_FOR_RATIO)

per_nta_recent = (
    recent_summary.sort_values(["nta_id", "mean_z", "residual_total"], ascending=[True, False, False])
    .groupby("nta_id", observed=True)
    .head(TOP_N_PER_NTA)
    .reset_index(drop=True)
)
per_nta_recent["rank_within_nta"] = per_nta_recent.groupby("nta_id", observed=True).cumcount() + 1

print("latest month:", latest_month.date())
print("recent window start:", recent_start.date())
display(overall_latest.head(30))
display(per_nta_latest.head(30))
display(per_nta_recent.head(30))

## Save Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

overall_latest_path = OUTPUT_DIR / "overall_latest_anomaly_rankings.csv"
per_nta_latest_path = OUTPUT_DIR / "per_nta_latest_anomaly_rankings.csv"
per_nta_recent_path = OUTPUT_DIR / "per_nta_recent_anomaly_rankings.csv"
model_scored_path = OUTPUT_DIR / "scored_model_grid.parquet"

overall_latest.to_csv(overall_latest_path, index=False)
per_nta_latest.to_csv(per_nta_latest_path, index=False)
per_nta_recent.to_csv(per_nta_recent_path, index=False)
model_df.to_parquet(model_scored_path, index=False)

print("Saved:")
print(" -", overall_latest_path)
print(" -", per_nta_latest_path)
print(" -", per_nta_recent_path)
print(" -", model_scored_path)